In [ ]:
import altair as alt
import polars as pl
import yaml

import common_functions


# Read config file. User should update this file path as needed
configfile = "/master/abagwell/workspace/github_project/variant-analysis/config/rhesus.yaml"
with open(configfile, 'r') as file:
    config = yaml.safe_load(file)

# Load colors
colors = pl.read_csv(config["colors"], separator="\t", schema_overrides={"Cohort": pl.String})

# Load cohorts
colonies_file = config["cohorts"]
colonies = pl.read_csv(colonies_file, separator="\t", comment_prefix="#", schema_overrides={"Id": pl.String}).select("Id", "Cohort").unique()

# Output path
output_dir = "/master/abagwell/variant-analysis/results/rhesus/inbreeding/plots"

# Seq type to filter on
seq_type = "WGS"

In [ ]:
# Only use when there subpops are in multiple files
# subsets = []
# for subset in config["subset"]:
#     ...
#     subsets.append(df)
# df = pl.concat(subsets)


input_file = config["results"] + f"inbreeding/GCTA/pass/{config['dataset']}.{config['subset']}.ibc"
indiv_col = "IID"
value_cols = ["NOMISS", "Fhat1", "Fhat2", "Fhat3"]
final_indiv_col = "Indiv"
final_value_col = "Fhat3"

df = common_functions.load_tsv(input_file, indiv_col, value_cols)
df2 = common_functions.df_with_trios(df, final_value_col)
df_with_parental_cohorts = common_functions.add_parental_cohorts(df2, final_indiv_col, final_value_col, df)


In [ ]:
boxplot = common_functions.plot_boxplot(df_with_parental_cohorts, final_value_col)
boxplot#.save(f'{output_dir}/P51_merging.{final_value_col}.boxplot.html')

In [ ]:
violinplot = common_functions.plot_violinplot(df, df_with_parental_cohorts, final_value_col)
violinplot#.save(f'{output_dir}/P51_merging.{final_value_col}.violinplot.html')

In [ ]:
lineplot =common_functions.plot_lineplot(df2, final_value_col, inverse=True)
lineplot#.save(f'{output_dir}/P51_merging.{final_value_col}.lineplot.html')

In [ ]:

# # Create theme
# #@alt.theme.register("black_marks", enable=True)
# def black_marks() -> alt.theme.ThemeConfig:
#     # return {
#     #     "config": {
#     #         "view": {"continuousWidth": 300, "continuousHeight": 300},
#     #         "mark": {"color": "black", "fill": "black"},
#     #     }
#     # }

#     # return alt.theme.ThemeConfig(
#     #     config = {
#     #         "bar": {"color": "red"}
#     #     }
#     # )


#     return {'spec': {'layer': [{'mark': {'type': 'area', 'orient': 'horizontal'},
#     'encoding': {'x': {'axis': {'labels': False,
#        'values': [0],
#        'grid': False,
#        'ticks': True},
#       'field': 'density',
#       'impute': None,
#       'scale': {'nice': False, 'zero': False},
#       'stack': 'center',
#       'title': None,
#       'type': 'quantitative'},
#      'y': {'field': 'Fhat2', 'title': 'Fhat2', 'type': 'quantitative'},},
#     'transform': [{'density': 'Fhat2',
#       'extent': [-0.3, 0.45],
#       'groupby': ['Cohort', 'color_idx'],
#       'as': ['Fhat2', 'density']}]},
#    {'mark': {'type': 'errorbar', 'extent': 'stderr'},
#     'encoding': {'y': {'field': 'Fhat2',
#       'title': 'Fhat2',
#       'type': 'quantitative'}}},
#    {'mark': {'type': 'circle', 'color': 'black'},
#     'encoding': {'y': {'aggregate': 'mean',
#       'field': 'Fhat2',
#       'title': 'Fhat2',
#       'type': 'quantitative'}}}],
#   'width': 92},
# }


    # return {
    #     "encoding": {
    #         "color": {
    #             "scale": {
    #                 "domain": ["Conventional source"],
    #                 "range": ["#A1C40F"]
    #             }
    #         }
    #     }
    # }

    # return {
    #     "spec": {
    #         "layer": [
    #         {
    #             "encoding": {
    #                 "color": {
    #                     "field": "Cohort",
    #                     "scale": {
    #                         "domain": ["Conventional source"],
    #                         "range": ["#A1C40F"]
    #                     },
    #                 }
    #             }
    #         }
    #         ]
    #     },
    # }

In [ ]:
unpivoted_df = df.unpivot(on=["Fhat1", "Fhat2", "Fhat3"], index=["color_idx", "Cohort"], variable_name="statistic", value_name="F")

In [ ]:
unpivoted_df

In [ ]:
alt.Chart(unpivoted_df).mark_boxplot().encode(
    alt.X("Cohort:N", title=None,
        sort=colors["Cohort"]
    ),
    alt.Y("F:Q", title="Inbreeding Coefficient, F"),
    alt.Column("statistic:N", 
               title="Inbreeding Statistic",
               #title=None
    ),
    #alt.Color("color_idx:N", legend=None).scale(scheme="category10"),
    alt.Color("Cohort:N", legend=None).scale(
        domain = list(colors["Cohort"]),
        range = list(colors["Color"])
    )
).properties(
    #title=["Inbreeding Coefficient", "Across Cohorts"]
).configure_title(
    anchor="middle"
)#.save('/master/abagwell/figures/final_plots/inbreeding.U42_WES.common_between_founding_cohorts2.all_Fs.all.html')

In [ ]:
# Just Fhat2
alt.Chart(unpivoted_df
    .filter(
        pl.col("statistic") == "Fhat2"
)).mark_boxplot().encode(
    alt.X("Cohort:N", title="Cohort",
        sort=colors["Cohort"]
    ),
    alt.Y("F:Q", title="Inbreeding Coefficient, Fhat2"),
    #alt.Color("color_idx:N", legend=None).scale(scheme="category10"),
    alt.Color("Cohort:N", legend=None).scale(
        domain = list(colors["Cohort"]),
        range = list(colors["Color"])
    )
).properties(
    title=["Inbreeding in Cohorts"]
).configure_title(
    anchor="middle"
)#.save('/master/abagwell/figures/final_plots/inbreeding.U42_WES.common_between_founding_cohorts2.Fhat2.html')

In [ ]:
unpivoted_df

In [ ]:
df

In [ ]:
from itertools import combinations

import pandas as pd
import scipy

# Do T-test for each pair of cohorts
results = []
for cohort1, cohort2 in combinations(list(df["Cohort"].unique()), 2):
    result = scipy.stats.ttest_ind(
        df.filter(
            pl.col('Cohort') == cohort1
    )['Fhat2'],
        df.filter(
            pl.col('Cohort') == cohort2
    )['Fhat2'],
    )
    results.append((cohort1, cohort2, result.statistic, result.pvalue, result.df))

stats_df = pd.DataFrame(results, columns=['Cohort1', 'Cohort2', 'Statistc', 'p-Value', 'DF'])
stats_df

In [ ]:
# Subset to living animals only
#living = pl.read_csv('/master/abagwell/workspace/living_rhesus_with_WES.tsv', separator='\t', schema_overrides={"Id": pl.String})
#df.join(living, how='inner', left_on='SAMPLE', right_on='Id').select(pl.mean('FRACTION'))